<a href="https://colab.research.google.com/github/edmundzen/dengue-early-warning/blob/main/member4_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install -q google-genai

**MEMBER 4 : Gemini AI & Integration**

Integrate the Gemini API.
Generate AI recommendations/explanations based on the risk scores.
Help connect all the modules and test the complete workflow.
## Tasks Completed
- Connected to Google BigQuery.
- Read the `inspection_priority` table generated by Member 2.
- Integrated the Gemini API.
- Generated AI recommendations for high-risk dengue areas.
- Saved the recommendations as a CSV (`inspection_with_ai.csv`).
- Verified the output for dashboard integration.

## Workflow
BigQuery (inspection_priority)
⬇
Gemini API
⬇
AI Recommendations
⬇
CSV Output
⬇
Looker Studio Dashboard

## Step 1: Import Required Libraries

In [12]:
from google import genai
from google.genai import types
import pandas as pd
from google.cloud import bigquery
from google.colab import auth

## Step 2: Authenticate with Google Cloud

In [33]:
auth.authenticate_user()

## Step 3: Connect to BigQuery

In [37]:
from google.cloud import bigquery

PROJECT_ID = "dengue-early-warning"

client = bigquery.Client(project=PROJECT_ID)

print("Connected to BigQuery!")

Connected to BigQuery!


## Step 4: Load the `inspection_priority` Table

In [15]:
query = """
SELECT *
FROM `dengue-early-warning.dengue_ew.inspection_priority`
LIMIT 10
"""

inspection_df = client.query(query).to_dataframe()

inspection_df.head()

,rank,cell_id,lat,lon,date,risk_score,top_factor
0,276176,1.25775_103.9275,1.25775,103.9275,2026-05-14,1.50,Case Density
1,290337,1.25775_103.9275,1.25775,103.9275,2026-04-19,1.35,Case Density
2,290768,1.25775_103.9275,1.25775,103.9275,2026-04-25,1.35,Case Density
3,293726,1.25775_103.9275,1.25775,103.9275,2026-04-14,1.26,Case Density
4,298480,1.25775_103.9275,1.25775,103.9275,2026-01-14,0.00,Case Density


## Step 5: Configure Gemini API

In [16]:
client_ai = genai.Client(api_key="API KEY")

In [17]:
sample = inspection_df.iloc[0]

In [18]:
sample = inspection_df.iloc[0]

prompt = f"""
You are a dengue control advisor.

Location: {sample['cell_id']}

Risk Score: {sample['risk_score']}

Top Risk Factor: {sample['top_factor']}

Based on this information:

1. Explain why this area is risky.
2. Suggest three inspection actions.
3. Mention whether inspection should be High, Medium or Low priority.

Keep your answer under 80 words.
"""

## Step 6: Upload to BigQuery

In [46]:
from google.cloud import bigquery

table_id = "dengue-early-warning.dengue_ew.gemini_recommendations"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    inspection_df,
    table_id,
    job_config=job_config
)

job.result()

print("✅ Gemini recommendations uploaded successfully!")

✅ Gemini recommendations uploaded successfully!


##Step 7: Verify that the table was created

In [36]:
query = """
SELECT *
FROM `dengue-early-warning.dengue_ew.gemini_recommendations`
LIMIT 5
"""

verify_df = client.query(query).to_dataframe()

verify_df.head()

,rank,cell_id,lat,lon,date,risk_score,top_factor
0,276176,1.25775_103.9275,1.25775,103.9275,2026-05-14,1.50,Case Density
1,290337,1.25775_103.9275,1.25775,103.9275,2026-04-19,1.35,Case Density
2,290768,1.25775_103.9275,1.25775,103.9275,2026-04-25,1.35,Case Density
3,293726,1.25775_103.9275,1.25775,103.9275,2026-04-14,1.26,Case Density
4,298480,1.25775_103.9275,1.25775,103.9275,2026-01-14,0.00,Case Density


In [19]:
response = client_ai.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)

print(response.text)

This area is risky due to high case density, indicating active dengue transmission nearby and increasing local infection risk.

**Inspection Actions:**
1.  Inspect all water containers (pails, discarded items).
2.  Check plant pot plates and trays for stagnant water.
3.  Clear clogged drains and gutters.

**Priority:** High.


## Step 8: Generate AI Recommendations

In [20]:
def generate_recommendation(row):

    prompt = f"""
    You are an experienced dengue public health officer.

    Location:
    {row['cell_id']}

    Risk Score:
    {row['risk_score']}

    Top Risk Factor:
    {row['top_factor']}

    Explain:
    1. Why this area is risky.
    2. Three preventive actions.
    3. Inspection priority.

    Keep it below 80 words.
    """

    response = client_ai.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [43]:
import pandas as pd

inspection_df = pd.read_csv("inspection_with_ai.csv")

In [41]:
print(inspection_df.columns.tolist())

['rank', 'cell_id', 'lat', 'lon', 'date', 'risk_score', 'top_factor']


In [45]:
import pandas as pd

inspection_df = pd.read_csv("inspection_with_ai.csv")

In [44]:
inspection_df.head()

,rank,cell_id,lat,lon,date,risk_score,top_factor,AI_Recommendation
0,276176,1.25775_103.9275,1.25775,103.9275,2026-05-14,1.50,Case Density,This Singapore location (1.25775_103.9275) is ...
1,290337,1.25775_103.9275,1.25775,103.9275,2026-04-19,1.35,Case Density,This area (1.25775_103.9275) carries a high ri...
2,290768,1.25775_103.9275,1.25775,103.9275,2026-04-25,1.35,Case Density,This area is high-risk (score 1.35) due to hig...
3,293726,1.25775_103.9275,1.25775,103.9275,2026-04-14,1.26,Case Density,This area (1.25775_103.9275) is high risk (1.2...
4,298480,1.25775_103.9275,1.25775,103.9275,2026-01-14,0.00,Case Density,This area currently has a very low dengue risk...


## Step 9: Export Recommendations to CSV

In [25]:
inspection_df.to_csv(
    "inspection_with_ai.csv",
    index=False
)

print("File saved successfully!")

File saved successfully!


In [26]:
inspection_df.to_csv("inspection_with_ai.csv", index=False)

print("CSV saved successfully!")

CSV saved successfully!


In [27]:
inspection_df["AI_Recommendation"].isnull().sum()

np.int64(0)

In [28]:
inspection_df[["risk_score","top_factor","AI_Recommendation"]].head(5)

,risk_score,top_factor,AI_Recommendation
0,1.50,Case Density,This Singapore location (1.25775_103.9275) is ...
1,1.35,Case Density,This area (1.25775_103.9275) carries a high ri...
2,1.35,Case Density,This area is high-risk (score 1.35) due to hig...
3,1.26,Case Density,This area (1.25775_103.9275) is high risk (1.2...
4,0.00,Case Density,This area currently has a very low dengue risk...


In [29]:
inspection_df.to_csv("inspection_with_ai.csv", index=False)

In [30]:
import os

os.listdir()

['.config', 'inspection_with_ai.csv', 'sample_data']

In [31]:
query = """
SELECT *
FROM `dengue-early-warning.dengue_ew.inspection_priority`
"""

In [32]:
inspection_df = client.query(query).to_dataframe()

# Conclusion

The Gemini AI module successfully converts dengue risk scores into actionable recommendations for health officials. These recommendations can be integrated into the dashboard to support faster and better decision-making during dengue outbreaks.